In [1]:
import random
import torch
import torch.nn as nn
from tensordict import TensorDict
from routerl import TrafficEnvironment

In [2]:
human_learning_episodes = 100

env_params = {
    "agent_parameters" : {
        "num_agents" : 128,
        "new_machines_after_mutation": 42,
        "human_parameters" : {
            "model" : "gawron"
        },
        "machine_parameters" : {
            "behavior" : "selfish",
        }
    },
    "simulator_parameters" : {
        "network_name" : "two_route_yield",
        # "network_name" : "arterial",
        # "sumo_type": "sumo-gui"
    },  
    "plotter_parameters" : {
        "phases" : [0, human_learning_episodes], # the number of episodes human learning will take
    },
}

In [3]:
env = TrafficEnvironment(seed=48, **env_params)

env.start()

for episode in range(human_learning_episodes):
    env.step()

env.mutation()

agent_to_id = {str(machine.id): i for i, machine in enumerate(env.machine_agents)}

[CONFIRMED] Environment variable exists: SUMO_HOME
[SUCCESS] Added module directory: /usr/share/sumo/tools
 Retrying in 1 seconds


In [4]:
# Simple net
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(len(env.machine_agents), 2)
        # self.linear = nn.Linear(2, 3)

    def forward(self, x):
        x = self.embedding(x)
        return x

simple_net = SimpleNet()

In [5]:
# Training
optim = torch.optim.SGD(simple_net.parameters(), lr=0.25)
exploration_eps = 0.7
n_epochs = 512

for epoch in range(n_epochs):
    cur_exploration_eps = exploration_eps * (1 - epoch / n_epochs)

    agent_action_output = {}

    optim.zero_grad()
    env.reset()
    for agent in env.agent_iter():
        observation, reward, termination, truncation, info = env.last()

        if termination or truncation:
            loss = (agent_action_output[agent] - reward) ** 2 / len(env.machine_agents)
            loss.backward()
            
            action = None
        else:
            q_table_tensor = simple_net(torch.tensor(agent_to_id[agent], dtype=torch.long))
            
            if random.random() < cur_exploration_eps:
                action = random.randrange(0, 2)
            else:
                q_table_list = q_table_tensor.tolist()
                action = q_table_list.index(max(q_table_list))
            
            agent_action_output[agent] = q_table_tensor[action]
        
        env.step(action)
    
    optim.step()


In [6]:
env.plot_results()

In [7]:
# Save model
torch.save(simple_net.state_dict(), 'weights.pth')

In [8]:
# Load model
simple_net.load_state_dict(torch.load('weights.pth'))

<All keys matched successfully>